# RDBScore V2 — Bedrock Execution and Results

This notebook contains only the experimental execution side of RDBScore V2: loading the benchmark instances generated by the exploration notebook, invoking Amazon Bedrock, consolidating model outputs, LLM-as-Judge adjudication, and final result aggregation.

It does **not** rediscover candidates or regenerate mutations.


## 1. Environment and Bedrock configuration


In [1]:
import os
import sys
import json
import time
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone

import yaml
import boto3
import botocore

print("Python:   ", sys.version)
print("boto3:    ", boto3.__version__)
print("botocore: ", botocore.__version__)


Python:    3.7.9 (default, Jun 24 2022, 18:14:15) 
[GCC 8.5.0 20210514 (Red Hat 8.5.0-13)]
boto3:     1.33.13
botocore:  1.33.13


### AWS credentials and clients

The following cell is preserved **exactly as provided in the source notebook**, as requested.


In [3]:
# AWS temporary credentials - fill locally before creating the Bedrock client
AWS_ACCESS_KEY_ID = "..."
AWS_SECRET_ACCESS_KEY = "..."
AWS_SESSION_TOKEN = "..."

bedrock = boto3.client(
    "bedrock-runtime",
    region_name="sa-east-1",
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    aws_session_token=AWS_SESSION_TOKEN
)

bedrock_control = boto3.client(
    "bedrock",
    region_name="sa-east-1",
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    aws_session_token=AWS_SESSION_TOKEN
)

print("Runtime:", bedrock.meta.service_model.service_name)
print("Control:", bedrock_control.meta.service_model.service_name)
print("Região:", bedrock.meta.region_name)

/opt/python37/lib/python3.7/site-packages/boto3/compat.py:82: PythonDeprecationWarning: Boto3 will no longer support Python 3.7 starting December 13, 2023. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.8 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


Runtime: bedrock-runtime
Control: bedrock
Região: sa-east-1


## 2. Load the frozen benchmark artifacts

Run the exploration/construction notebook first. This execution notebook consumes its manifests and generated audit inputs.


In [4]:
MODEL_ID = "anthropic.claude-3-sonnet-20240229-v1:0"

body = {
    "anthropic_version": "bedrock-2023-05-31",
    "max_tokens": 10,
    "temperature": 0,
    "messages": [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "Reply with exactly: OK"
                }
            ]
        }
    ]
}

response = bedrock.invoke_model(
    modelId=MODEL_ID,
    body=json.dumps(body),
    contentType="application/json",
    accept="application/json"
)

result = json.loads(response["body"].read())

print(result)

{'id': 'msg_bdrk_018JWuh9miDwhEDC6EU6n7Fi', 'type': 'message', 'role': 'assistant', 'model': 'claude-3-sonnet-20240229', 'content': [{'type': 'text', 'text': 'OK'}], 'stop_reason': 'end_turn', 'stop_sequence': None, 'usage': {'input_tokens': 12, 'output_tokens': 4}}


In [5]:
# Enter the Spider2 working directory when starting from the parent directory.
if Path("Spider2").exists() and not Path("spider2-dbt").exists():
    os.chdir("Spider2")

RDBSCORE_OUTPUT_V2 = Path("rdbscore_generated_v2")
manifest_v2_path = RDBSCORE_OUTPUT_V2 / "generation_manifest.json"
manifest_l3_path = RDBSCORE_OUTPUT_V2 / "L3" / "generation_manifest_l3.json"

for required in [manifest_v2_path, manifest_l3_path]:
    if not required.exists():
        raise FileNotFoundError(
            "Missing benchmark artifact: {}. Run the exploration/construction notebook first.".format(required)
        )

generation_manifest_v2 = json.loads(manifest_v2_path.read_text(encoding="utf-8"))
generation_manifest_l3 = json.loads(manifest_l3_path.read_text(encoding="utf-8"))

print("L1/L2 manifest records:", len(generation_manifest_v2))
print("L3 manifest records:   ", len(generation_manifest_l3))


L1/L2 manifest records: 1510
L3 manifest records:    51


## 3. Claude Sonnet audit protocol — L1/L2


In [6]:
# ============================================================
# V2 prompts — one fixed prompt per evidence level
# ============================================================

RDBSCORE_L1_PROMPT_V2 = """
You are auditing the structural integrity of a relational database schema.

Analyze the provided database artifacts and identify any structural schema
problems that can be justified directly from the available evidence.

Consider the schema definitions, declared constraints, relationships,
column properties, and SQL definitions when relevant.

For each problem found, report:

1. defect_type
2. affected_element
3. description
4. evidence
5. reasoning

Report a problem only when it is supported by the provided artifacts.
Do not invent constraints or requirements that are not supported by the evidence.

If no justified structural schema problem is found, return an empty findings list.

Return only valid JSON using the following structure:

{{
  "findings": [
    {{
      "defect_type": "...",
      "affected_element": "...",
      "description": "...",
      "evidence": "...",
      "reasoning": "..."
    }}
  ]
}}

DATABASE ARTIFACTS

{audit_input}
"""


RDBSCORE_L2_PROMPT_V2 = """
You are auditing the normalization design of a relational database schema.

Analyze the provided schema artifacts and identify any normalization-related
design problems that can be justified from the available evidence.

Consider how keys, attributes, dependencies, relationships, and relation
structures interact. Some design problems may require reasoning across
multiple schema elements rather than inspecting a single constraint or
column in isolation.

For each problem found, report:

1. defect_type
2. affected_element
3. description
4. evidence
5. reasoning

Report a problem only when it can be justified from the provided artifacts.
Do not invent dependencies, constraints, or domain semantics that are not
supported by the evidence.

If no justified normalization-related design problem is found, return an empty findings list.

Return only valid JSON using the following structure:

{{
  "findings": [
    {{
      "defect_type": "...",
      "affected_element": "...",
      "description": "...",
      "evidence": "...",
      "reasoning": "..."
    }}
  ]
}}

DATABASE ARTIFACTS

{audit_input}
"""


def prompt_for_level_v2(level, audit_input):
    if level == "L1":
        return RDBSCORE_L1_PROMPT_V2.format(audit_input=audit_input), "RDBSCORE_L1_PROMPT_V2"
    if level == "L2":
        return RDBSCORE_L2_PROMPT_V2.format(audit_input=audit_input), "RDBSCORE_L2_PROMPT_V2"
    raise ValueError("Unsupported benchmark level: {}".format(level))


print("V2 prompts ready.")


V2 prompts ready.


In [7]:
# ============================================================
# V2 Bedrock invocation
# ============================================================

import json
import time
from datetime import datetime, timezone


def invoke_bedrock_audit_v2(prompt, prompt_version, max_tokens=1200, temperature=0.0):
    request_body = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": max_tokens,
        "temperature": temperature,
        "messages": [{"role": "user", "content": prompt}]
    }
    started_at = datetime.now(timezone.utc).isoformat()
    t0 = time.perf_counter()

    try:
        response = bedrock.invoke_model(
            modelId=MODEL_ID,
            body=json.dumps(request_body),
            contentType="application/json",
            accept="application/json"
        )
        latency_seconds = time.perf_counter() - t0
        result = json.loads(response["body"].read())
        usage = result.get("usage", {})
        input_tokens = usage.get("input_tokens")
        output_tokens = usage.get("output_tokens")
        total_tokens = (input_tokens + output_tokens) if input_tokens is not None and output_tokens is not None else None
        output_text = "\n".join(
            block.get("text", "") for block in result.get("content", [])
            if block.get("type") == "text"
        )
        parsed_output = None
        valid_json = False
        try:
            parsed_output = json.loads(output_text)
            valid_json = isinstance(parsed_output, dict) and isinstance(parsed_output.get("findings"), list)
        except Exception:
            pass

        return {
            "status": "SUCCESS",
            "model": {"model_id": MODEL_ID, "temperature": temperature, "max_tokens": max_tokens},
            "prompt": {"version": prompt_version},
            "execution": {"started_at_utc": started_at, "latency_seconds": latency_seconds},
            "usage": {"input_tokens": input_tokens, "output_tokens": output_tokens, "total_tokens": total_tokens},
            "response": {
                "text": output_text,
                "valid_json": valid_json,
                "parsed": parsed_output,
                "stop_reason": result.get("stop_reason")
            }
        }
    except Exception as e:
        return {
            "status": "ERROR",
            "model": {"model_id": MODEL_ID, "temperature": temperature, "max_tokens": max_tokens},
            "prompt": {"version": prompt_version},
            "execution": {"started_at_utc": started_at, "latency_seconds": time.perf_counter() - t0},
            "error": {"type": type(e).__name__, "message": str(e)}
        }


In [8]:
# ============================================================
# V2 full Bedrock batch — checkpoint/resume
# ============================================================


def run_rdbscore_bedrock_batch_v2(generation_manifest, output_root, delay_seconds=0.0):
    output_root = Path(output_root)
    started = datetime.now(timezone.utc).isoformat()
    records = [r for r in generation_manifest if r.get("status") == "GENERATED"]
    succeeded = failed = resumed = 0
    total_input = total_output = total_tokens = 0
    total_latency = 0.0

    print("=" * 80)
    print("RDBScore V2 Bedrock Execution")
    print("=" * 80)
    print("Instances:", len(records))
    print("Model:", MODEL_ID)
    print("L1 prompt: RDBSCORE_L1_PROMPT_V2")
    print("L2 prompt: RDBSCORE_L2_PROMPT_V2")
    print("=" * 80)

    for i, record in enumerate(records, 1):
        mutation_dir = Path(record["output_directory"])
        result_path = mutation_dir / "llm_result_v2.json"

        if result_path.exists():
            try:
                existing = json.loads(result_path.read_text(encoding="utf-8"))
                if existing.get("status") == "SUCCESS":
                    resumed += 1; succeeded += 1
                    u = existing.get("usage", {}); e = existing.get("execution", {})
                    total_input += u.get("input_tokens") or 0
                    total_output += u.get("output_tokens") or 0
                    total_tokens += u.get("total_tokens") or 0
                    total_latency += e.get("latency_seconds") or 0.0
                    print("[{}/{}] RESUME  {}".format(i, len(records), record["mutation_id"]))
                    continue
            except Exception:
                pass

        try:
            audit_input = (mutation_dir / "audit_input.txt").read_text(encoding="utf-8")
            prompt, prompt_version = prompt_for_level_v2(record["level"], audit_input)
            result = invoke_bedrock_audit_v2(prompt, prompt_version, max_tokens=1200, temperature=0.0)
            result["benchmark"] = {
                "mutation_id": record["mutation_id"],
                "operator_id": record["operator_id"],
                "level": record["level"],
                "defect_family": record["defect_family"],
                "project": record["project"],
                "artifact": record["artifact"],
                "protocol_version": "RDBSCORE_V2"
            }
            result_path.write_text(json.dumps(result, indent=2, default=str), encoding="utf-8")

            if result.get("status") == "SUCCESS":
                succeeded += 1
                u = result.get("usage", {}); e = result.get("execution", {})
                it = u.get("input_tokens") or 0; ot = u.get("output_tokens") or 0; tt = u.get("total_tokens") or 0
                lat = e.get("latency_seconds") or 0.0
                total_input += it; total_output += ot; total_tokens += tt; total_latency += lat
                print("[{}/{}] OK      {} | {} + {} = {} tokens | {:.3f}s".format(
                    i, len(records), record["mutation_id"], it, ot, tt, lat))
            else:
                failed += 1
                print("[{}/{}] ERROR   {} | {}".format(
                    i, len(records), record["mutation_id"], result.get("error", {}).get("message", "Unknown error")))
        except Exception as e:
            failed += 1
            print("[{}/{}] INPUT ERROR {} | {}".format(i, len(records), record["mutation_id"], str(e)))

        if delay_seconds > 0:
            time.sleep(delay_seconds)

    summary = {
        "protocol_version": "RDBSCORE_V2",
        "started_at_utc": started,
        "finished_at_utc": datetime.now(timezone.utc).isoformat(),
        "model_id": MODEL_ID,
        "prompt_versions": {"L1": "RDBSCORE_L1_PROMPT_V2", "L2": "RDBSCORE_L2_PROMPT_V2"},
        "instances": len(records), "succeeded": succeeded, "failed": failed, "resumed": resumed,
        "usage": {
            "input_tokens": total_input, "output_tokens": total_output, "total_tokens": total_tokens,
            "average_tokens_per_instance": total_tokens / succeeded if succeeded else None
        },
        "latency": {
            "total_seconds": total_latency,
            "average_seconds": total_latency / succeeded if succeeded else None
        }
    }
    summary_path = output_root / "bedrock_batch_summary_v2.json"
    summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
    print("\n" + "=" * 80 + "\nBATCH COMPLETE\n" + "=" * 80)
    print(json.dumps(summary, indent=2))
    print("\nSummary saved to:", summary_path)
    return summary


### Execute/resume L1/L2 batch


In [9]:
# Execute only after inspecting applicability, generated mutations,
# mutation-opacity audit, and audit_input.txt samples.

bedrock_batch_summary_v2 = run_rdbscore_bedrock_batch_v2(
    generation_manifest_v2,
    RDBSCORE_OUTPUT_V2,
    delay_seconds=0.0
)

RDBScore V2 Bedrock Execution
Instances: 1510
Model: anthropic.claude-3-sonnet-20240229-v1:0
L1 prompt: RDBSCORE_L1_PROMPT_V2
L2 prompt: RDBSCORE_L2_PROMPT_V2
[1/1510] OK      L1_REMOVE_UNIQUE_79a9c4517fe5 | 2368 + 55 = 2423 tokens | 2.494s
[2/1510] OK      L1_REMOVE_UNIQUE_afbd2637755f | 2368 + 54 = 2422 tokens | 2.389s
[3/1510] OK      L1_REMOVE_UNIQUE_b5f05109403c | 2368 + 55 = 2423 tokens | 2.654s
[4/1510] OK      L1_REMOVE_UNIQUE_18920c375277 | 2368 + 54 = 2422 tokens | 2.551s
[5/1510] OK      L1_REMOVE_UNIQUE_34e927ecbd1a | 2368 + 58 = 2426 tokens | 2.384s
[6/1510] OK      L1_REMOVE_UNIQUE_eab6d76cac34 | 2368 + 58 = 2426 tokens | 2.410s
[7/1510] OK      L1_REMOVE_UNIQUE_a5cc68f55184 | 2368 + 58 = 2426 tokens | 2.621s
[8/1510] OK      L1_REMOVE_UNIQUE_de446942a189 | 2630 + 58 = 2688 tokens | 2.403s
[9/1510] OK      L1_REMOVE_UNIQUE_701325e33063 | 2630 + 58 = 2688 tokens | 2.483s
[10/1510] OK      L1_REMOVE_UNIQUE_2eb44333e5b4 | 2630 + 54 = 2684 tokens | 2.516s
[11/1510] OK      L1

## 4. Consolidate L1/L2 outputs


In [10]:
# ============================================================
# RDBScore V2 — Consolidated result extraction
# ============================================================

import json
from pathlib import Path
from collections import Counter, defaultdict


def extract_balanced_json(text):
    """Conservative extraction of the first balanced JSON object."""
    if not text:
        return None

    start = text.find("{")
    if start < 0:
        return None

    depth = 0
    in_string = False
    escape = False

    for i in range(start, len(text)):
        ch = text[i]

        if in_string:
            if escape:
                escape = False
            elif ch == "\\":
                escape = True
            elif ch == '"':
                in_string = False
            continue

        if ch == '"':
            in_string = True
        elif ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1

            if depth == 0:
                candidate = text[start:i + 1]
                try:
                    return json.loads(candidate)
                except Exception:
                    return None

    return None


records_v2 = []

for manifest in generation_manifest_v2:

    out_dir = Path(manifest["output_directory"])
    gt_path = out_dir / "ground_truth.json"
    result_path = out_dir / "llm_result_v2.json"

    gt = json.loads(gt_path.read_text(encoding="utf-8"))
    result = json.loads(result_path.read_text(encoding="utf-8"))

    # --------------------------------------------------------
    # Normalize result layout
    # --------------------------------------------------------

    if "response" in result:
        # Current L1/L2 result format
        response = result.get("response", {})
        text = response.get("text", "")
        parsed = response.get("parsed")

    elif "llm_result_v2" in result:
        # Compatibility with compact/older result format
        response = result.get("llm_result_v2", {})
        text = response.get("text", "")
        parsed = response.get("parsed")

    else:
        response = {}
        text = ""
        parsed = None

    if isinstance(parsed, dict):
        recovery_status = "STRICT_JSON"
    else:
        parsed = extract_balanced_json(text)

        if isinstance(parsed, dict):
            recovery_status = "EXTRACTED_JSON"
        else:
            recovery_status = "NO_JSON"

    findings = []

    if isinstance(parsed, dict):
        candidate_findings = parsed.get("findings", [])
        if isinstance(candidate_findings, list):
            findings = candidate_findings

        # --------------------------------------------------------
    # Execution metadata
    #
    # Historical V2 result files may use slightly different
    # layouts. These fields are informative only and must not
    # prevent benchmark consolidation.
    # --------------------------------------------------------

    usage = result.get("usage", {})
    execution = result.get("execution", {})

    # Compatibility with older result layout, if present.
    if not usage:
        usage = (
            result
            .get("llm_result_v2", {})
        )

    if not execution:
        execution = (
            result
            .get("llm_result_v2", {})
        )

    records_v2.append({
        "mutation_id": gt["mutation_id"],
        "operator_id": gt["operator_id"],
        "level": gt["level"],
        "defect_family": gt["defect_family"],
        "expected_defect": gt["expected_defect"],
        "project": gt["project"],
        "artifact": gt["artifact"],
        "candidate": gt["candidate"],

        "status": result.get("status"),
        "recovery_status": recovery_status,

        "findings": findings,
        "finding_count": len(findings),

        "input_tokens": usage.get(
            "input_tokens"
        ),

        "output_tokens": usage.get(
            "output_tokens"
        ),

        "latency_seconds": execution.get(
            "latency_seconds"
        )
    })

# ============================================================
# Basic integrity
# ============================================================

print("=" * 90)
print("RDBSCORE V2 — CONSOLIDATED RESULTS")
print("=" * 90)

print("\nInstances:", len(records_v2))

print("\nJSON recovery:")
for k, v in Counter(r["recovery_status"] for r in records_v2).items():
    print(f"  {k:<20} {v:>5}")

print("\nBy operator:")

for operator_id in sorted(set(r["operator_id"] for r in records_v2)):

    rows = [
        r for r in records_v2
        if r["operator_id"] == operator_id
    ]

    nonempty = sum(r["finding_count"] > 0 for r in rows)
    findings = sum(r["finding_count"] for r in rows)
    recoverable = sum(r["recovery_status"] != "NO_JSON" for r in rows)

    print(
        f"{operator_id:<45} "
        f"N={len(rows):>4} | "
        f"recoverable={recoverable:>4} | "
        f"nonempty={nonempty:>4} "
        f"({nonempty / len(rows):6.1%}) | "
        f"findings={findings:>5}"
    )


# ============================================================
# Save consolidated dataset
# ============================================================

output_path = Path("rdbscore_experiment_v2_consolidated.json")

output_path.write_text(
    json.dumps(
        records_v2,
        indent=2,
        ensure_ascii=False,
        default=str
    ),
    encoding="utf-8"
)

print("\nSaved:", output_path)

RDBSCORE V2 — CONSOLIDATED RESULTS

Instances: 1510

JSON recovery:
  EXTRACTED_JSON        1125
  STRICT_JSON            312
  NO_JSON                 73

By operator:
L1_REMOVE_NOT_NULL                            N= 861 | recoverable= 791 | nonempty= 160 ( 18.6%) | findings=  297
L1_REMOVE_UNIQUE                              N= 300 | recoverable= 297 | nonempty=  38 ( 12.7%) | findings=   70
L2_INTRODUCE_TRANSITIVE_DEPENDENCY            N= 349 | recoverable= 349 | nonempty= 343 ( 98.3%) | findings=  569

Saved: rdbscore_experiment_v2_consolidated.json


## 5. Haiku adjudication — L1/L2


In [11]:
def invoke_haiku_judge(record):

    payload = {
        "operator_id": record["operator_id"],
        "level": record["level"],
        "expected_defect": record["expected_defect"],
        "candidate": record["candidate"],
        "findings": record["findings"]
    }

    body = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 500,
        "temperature": 0,
        "system": JUDGE_SYSTEM,
        "messages": [
            {
                "role": "user",
                "content": json.dumps(
                    payload,
                    ensure_ascii=False
                )
            }
        ]
    }

    t0 = time.time()

    response = bedrock.invoke_model(
        modelId=JUDGE_MODEL_ID,
        body=json.dumps(body),
        contentType="application/json",
        accept="application/json"
    )

    result = json.loads(
        response["body"].read()
    )

    text = "".join(
        block.get("text", "")
        for block in result.get("content", [])
        if block.get("type") == "text"
    )

    parsed, recovery = recover_judge_json(text)

    return {
        "parsed": parsed,
        "raw_response": text,
        "json_recovery": recovery,
        "usage": result.get("usage", {}),
        "latency_seconds": time.time() - t0
    }

In [12]:
# ============================================================
# RDBScore V2 — Haiku Judge dependencies
# ============================================================

JUDGE_MODEL_ID = "anthropic.claude-3-haiku-20240307-v1:0"
JUDGE_PROMPT_VERSION = "RDBSCORE_JUDGE_V1"

JUDGE_SYSTEM = """
You are an independent evaluator for a database schema auditing benchmark.

Your task is to determine whether the findings produced by another language
model detect the specific defect injected by the benchmark.

You are NOT auditing the schema independently.

You will receive:
- the benchmark level;
- the mutation operator;
- the expected injected defect;
- the schema element affected by the mutation;
- the findings produced by the audited model.

Classify the response as exactly one of:

MATCH:
At least one finding correctly identifies the injected defect and the
relevant affected schema element. Equivalent terminology is acceptable.

PARTIAL_MATCH:
At least one finding recognizes the general injected problem or the affected
element, but omits, confuses, or incorrectly states an important element
needed to establish the injected defect.

NO_MATCH:
None of the findings identifies the injected defect.

Additional findings unrelated to the injected defect do not affect the
classification.

Do not require exact wording or the benchmark's defect label when the
model's finding is semantically equivalent.

Return ONLY valid JSON with this structure:

{
  "classification": "MATCH | PARTIAL_MATCH | NO_MATCH",
  "matched_finding_index": null,
  "justification": "brief explanation"
}
"""


def recover_judge_json(text):

    text = (text or "").strip()

    # 1. Strict JSON
    try:
        obj = json.loads(text)

        if (
            isinstance(obj, dict)
            and obj.get("classification") in {
                "MATCH",
                "PARTIAL_MATCH",
                "NO_MATCH"
            }
        ):
            return obj, "STRICT_JSON"

    except (ValueError, TypeError):
        pass

    # 2. Recover JSON embedded in additional text
    decoder = json.JSONDecoder()

    for i, char in enumerate(text):

        if char != "{":
            continue

        try:
            obj, _ = decoder.raw_decode(text[i:])

            if (
                isinstance(obj, dict)
                and obj.get("classification") in {
                    "MATCH",
                    "PARTIAL_MATCH",
                    "NO_MATCH"
                }
            ):
                return obj, "EXTRACTED_JSON"

        except (ValueError, TypeError):
            continue

    return None, "NO_JSON"

In [13]:
# ============================================================
# RDBScore V2 — Haiku Judge Batch
# Judge only instances containing findings
# Incremental + resumable execution
# ============================================================

import json
import time
from pathlib import Path
from collections import Counter

JUDGE_OUTPUT_DIR = Path("rdbscore_generated_v2/judge_haiku_v1")
JUDGE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

JUDGE_SUMMARY_FILE = (
    JUDGE_OUTPUT_DIR / "judge_batch_summary.json"
)

# Only cases where the audited model produced findings
cases_to_judge = [
    r for r in records_v2
    if r["finding_count"] > 0
]

print("Cases requiring judge:", len(cases_to_judge))


# ------------------------------------------------------------
# Run judge
# ------------------------------------------------------------

started_at = time.time()

succeeded = 0
failed = 0
resumed = 0

input_tokens = 0
output_tokens = 0

for i, record in enumerate(cases_to_judge, start=1):

    mutation_id = record["mutation_id"]

    output_file = (
        JUDGE_OUTPUT_DIR /
        f"{mutation_id}.judge.json"
    )

    # --------------------------------------------------------
    # Resume support
    # --------------------------------------------------------

    if output_file.exists():
        try:
            previous = json.loads(
                output_file.read_text(encoding="utf-8")
            )

            if previous.get("status") == "SUCCESS":
                resumed += 1

                usage = previous.get("judge", {}).get(
                    "usage", {}
                )

                input_tokens += usage.get(
                    "input_tokens", 0
                )

                output_tokens += usage.get(
                    "output_tokens", 0
                )

                if i % 25 == 0:
                    print(
                        f"[{i:>3}/{len(cases_to_judge)}] "
                        f"resume | "
                        f"success={succeeded} "
                        f"failed={failed} "
                        f"resumed={resumed}"
                    )

                continue

        except Exception:
            pass

    # --------------------------------------------------------
    # Judge
    # --------------------------------------------------------

    try:

        judge = invoke_haiku_judge(record)

        parsed = judge.get("parsed")

        valid_classification = (
            isinstance(parsed, dict)
            and parsed.get("classification")
            in {
                "MATCH",
                "PARTIAL_MATCH",
                "NO_MATCH"
            }
        )

        status = (
            "SUCCESS"
            if valid_classification
            else "INVALID_RESPONSE"
        )

        result = {
            "status": status,

            "mutation_id": mutation_id,
            "operator_id": record["operator_id"],
            "level": record["level"],
            "project": record["project"],
            "artifact": record["artifact"],

            "ground_truth": {
                "expected_defect":
                    record["expected_defect"],
                "candidate":
                    record["candidate"]
            },

            "model_findings":
                record["findings"],

            "judge": judge
        }

        output_file.write_text(
            json.dumps(
                result,
                indent=2,
                ensure_ascii=False,
                default=str
            ),
            encoding="utf-8"
        )

        if status == "SUCCESS":
            succeeded += 1
        else:
            failed += 1

        usage = judge.get("usage", {})

        input_tokens += usage.get(
            "input_tokens", 0
        )

        output_tokens += usage.get(
            "output_tokens", 0
        )

    except Exception as e:

        failed += 1

        error_result = {
            "status": "ERROR",
            "mutation_id": mutation_id,
            "operator_id": record["operator_id"],
            "error": repr(e)
        }

        output_file.write_text(
            json.dumps(
                error_result,
                indent=2,
                ensure_ascii=False
            ),
            encoding="utf-8"
        )

        # TEMPORARY DEBUG — stop on first error
        print("\nJUDGE ERROR:")
        print(type(e).__name__)
        print(str(e))
        print("\nRecord:")
        print(json.dumps(record, indent=2, ensure_ascii=False))

        raise

    # --------------------------------------------------------
    # Progress
    # --------------------------------------------------------

    if (
        i == 1
        or i % 25 == 0
        or i == len(cases_to_judge)
    ):
        print(
            f"[{i:>3}/{len(cases_to_judge)}] "
            f"success={succeeded} "
            f"failed={failed} "
            f"resumed={resumed}"
        )


# ------------------------------------------------------------
# Consolidate classifications
# ------------------------------------------------------------

judge_records = []

for record in cases_to_judge:

    path = (
        JUDGE_OUTPUT_DIR /
        f"{record['mutation_id']}.judge.json"
    )

    if not path.exists():
        continue

    data = json.loads(
        path.read_text(encoding="utf-8")
    )

    if data.get("status") != "SUCCESS":
        continue

    parsed = data["judge"]["parsed"]

    judge_records.append({
        "mutation_id":
            record["mutation_id"],

        "operator_id":
            record["operator_id"],

        "level":
            record["level"],

        "classification":
            parsed["classification"],

        "matched_finding_index":
            parsed.get("matched_finding_index"),

        "justification":
            parsed.get("justification")
    })


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

elapsed = time.time() - started_at

print("\n" + "=" * 100)
print("RDBSCORE V2 — HAIKU JUDGE SUMMARY")
print("=" * 100)

print("\nCases with findings :", len(cases_to_judge))
print("Judge records       :", len(judge_records))
print("Succeeded           :", succeeded)
print("Failed              :", failed)
print("Resumed             :", resumed)

print("\nOverall classifications:")

overall = Counter(
    r["classification"]
    for r in judge_records
)

for classification in [
    "MATCH",
    "PARTIAL_MATCH",
    "NO_MATCH"
]:
    n = overall.get(classification, 0)

    print(
        f"  {classification:<15} "
        f"{n:>4} "
        f"({n / len(judge_records):6.1%})"
    )


print("\nBy operator:")

operators = sorted(
    set(
        r["operator_id"]
        for r in judge_records
    )
)

for operator_id in operators:

    rows = [
        r for r in judge_records
        if r["operator_id"] == operator_id
    ]

    counts = Counter(
        r["classification"]
        for r in rows
    )

    print("\n", operator_id)

    for classification in [
        "MATCH",
        "PARTIAL_MATCH",
        "NO_MATCH"
    ]:

        n = counts.get(classification, 0)

        print(
            f"    {classification:<15} "
            f"{n:>4} "
            f"({n / len(rows):6.1%})"
        )


print("\nUsage:")
print("  Input tokens :", input_tokens)
print("  Output tokens:", output_tokens)
print("  Total tokens :", input_tokens + output_tokens)

print(
    f"\nElapsed: {elapsed / 60:.1f} minutes"
)


# ------------------------------------------------------------
# Save summary
# ------------------------------------------------------------

summary = {
    "judge_model": JUDGE_MODEL_ID,
    "prompt_version": JUDGE_PROMPT_VERSION,
    "cases_with_findings": len(cases_to_judge),
    "judge_records": len(judge_records),
    "succeeded": succeeded,
    "failed": failed,
    "resumed": resumed,
    "classifications": dict(overall),
    "input_tokens": input_tokens,
    "output_tokens": output_tokens,
    "elapsed_seconds": elapsed
}

JUDGE_SUMMARY_FILE.write_text(
    json.dumps(
        summary,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print("\nSummary saved to:", JUDGE_SUMMARY_FILE)

Cases requiring judge: 541
[  1/541] success=1 failed=0 resumed=0
[ 25/541] success=25 failed=0 resumed=0
[ 50/541] success=50 failed=0 resumed=0
[ 75/541] success=75 failed=0 resumed=0
[100/541] success=100 failed=0 resumed=0
[125/541] success=125 failed=0 resumed=0
[150/541] success=150 failed=0 resumed=0
[175/541] success=175 failed=0 resumed=0
[200/541] success=200 failed=0 resumed=0
[225/541] success=225 failed=0 resumed=0
[250/541] success=250 failed=0 resumed=0
[275/541] success=275 failed=0 resumed=0
[300/541] success=300 failed=0 resumed=0
[325/541] success=325 failed=0 resumed=0
[350/541] success=350 failed=0 resumed=0
[375/541] success=375 failed=0 resumed=0
[400/541] success=400 failed=0 resumed=0
[425/541] success=425 failed=0 resumed=0
[450/541] success=450 failed=0 resumed=0
[475/541] success=475 failed=0 resumed=0
[500/541] success=500 failed=0 resumed=0
[525/541] success=525 failed=0 resumed=0
[541/541] success=541 failed=0 resumed=0

RDBSCORE V2 — HAIKU JUDGE SUMMARY


## 6. Claude Sonnet audit protocol — L3


In [14]:
# ============================================================
# RDBScore V2 — Execute L3 only with Claude 3 Sonnet
# ============================================================

import json
import time
from datetime import datetime, timezone
from pathlib import Path


L3_MODEL_ID = "anthropic.claude-3-sonnet-20240229-v1:0"

L3_RESULTS_DIR = Path(
    "rdbscore_generated_v2/L3/bedrock_results"
)
L3_RESULTS_DIR.mkdir(parents=True, exist_ok=True)


def invoke_claude_l3(prompt):

    body = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 1200,
        "temperature": 0,
        "messages": [
            {
                "role": "user",
                "content": prompt
            }
        ]
    }

    t0 = time.time()

    response = bedrock.invoke_model(
        modelId=L3_MODEL_ID,
        body=json.dumps(body),
        contentType="application/json",
        accept="application/json"
    )

    latency = time.time() - t0

    payload = json.loads(
        response["body"].read()
    )

    text = ""

    for block in payload.get("content", []):
        if block.get("type") == "text":
            text += block.get("text", "")

    usage = payload.get("usage", {})

    return {
        "text": text,
        "input_tokens":
            usage.get("input_tokens", 0),
        "output_tokens":
            usage.get("output_tokens", 0),
        "latency_seconds":
            latency
    }


# ------------------------------------------------------------
# Execute
# ------------------------------------------------------------

started = datetime.now(
    timezone.utc
).isoformat()

records_l3 = []

total_input = 0
total_output = 0
total_latency = 0.0

succeeded = 0
failed = 0


for i, instance in enumerate(
    generation_manifest_l3,
    start=1
):

    instance_id = instance["instance_id"]

    result_path = (
        L3_RESULTS_DIR
        /
        "{}.json".format(instance_id)
    )

    print(
        "[{:02d}/{}] {}".format(
            i,
            len(generation_manifest_l3),
            instance_id
        ),
        end=" ... "
    )

    try:

        prompt = Path(
            instance["audit_input_path"]
        ).read_text(
            encoding="utf-8"
        )

        result = invoke_claude_l3(
            prompt
        )

        record = {
            **instance,

            "model_id":
                L3_MODEL_ID,

            "prompt_version":
                "RDBSCORE_L3_PROMPT_V2",

            "llm_result_v2": result,

            "status":
                "SUCCESS"
        }

        result_path.write_text(
            json.dumps(
                record,
                indent=2,
                ensure_ascii=False
            ),
            encoding="utf-8"
        )

        records_l3.append(record)

        total_input += result[
            "input_tokens"
        ]

        total_output += result[
            "output_tokens"
        ]

        total_latency += result[
            "latency_seconds"
        ]

        succeeded += 1

        print(
            "OK | {} + {} tokens".format(
                result["input_tokens"],
                result["output_tokens"]
            )
        )

    except Exception as e:

        failed += 1

        record = {
            **instance,

            "model_id":
                L3_MODEL_ID,

            "status":
                "ERROR",

            "error":
                str(e)
        }

        records_l3.append(record)

        result_path.write_text(
            json.dumps(
                record,
                indent=2,
                ensure_ascii=False
            ),
            encoding="utf-8"
        )

        print(
            "ERROR:",
            str(e)
        )


finished = datetime.now(
    timezone.utc
).isoformat()


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

summary_l3_bedrock = {

    "protocol_version":
        "RDBSCORE_V2",

    "level":
        "L3",

    "started_at_utc":
        started,

    "finished_at_utc":
        finished,

    "model_id":
        L3_MODEL_ID,

    "prompt_version":
        "RDBSCORE_L3_PROMPT_V2",

    "instances":
        len(generation_manifest_l3),

    "succeeded":
        succeeded,

    "failed":
        failed,

    "usage": {
        "input_tokens":
            total_input,

        "output_tokens":
            total_output,

        "total_tokens":
            total_input + total_output,

        "average_tokens_per_instance":
            (
                (total_input + total_output)
                / succeeded
                if succeeded
                else None
            )
    },

    "latency": {
        "total_seconds":
            total_latency,

        "average_seconds":
            (
                total_latency / succeeded
                if succeeded
                else None
            )
    }
}


summary_path = (
    L3_RESULTS_DIR
    /
    "bedrock_batch_summary_l3.json"
)

summary_path.write_text(
    json.dumps(
        summary_l3_bedrock,
        indent=2
    ),
    encoding="utf-8"
)


print("\n" + "=" * 100)
print("L3 BEDROCK BATCH COMPLETE")
print("=" * 100)

print(
    json.dumps(
        summary_l3_bedrock,
        indent=2
    )
)

print(
    "\nSummary saved:",
    summary_path
)

[01/51] L3_REL_0001_airbnb001_dim_listings_hosts_host_id ... OK | 864 + 151 tokens
[02/51] L3_REL_0002_airbnb001_dim_listings_hosts_listing_id ... OK | 858 + 129 tokens
[03/51] L3_REL_0003_airbnb001_fct_reviews_listing_id ... OK | 863 + 265 tokens
[04/51] L3_REL_0004_airbnb001_fct_reviews_listing_id ... OK | 861 + 174 tokens
[05/51] L3_REL_0005_airbnb002_dim_listings_hosts_host_id ... OK | 535 + 260 tokens
[06/51] L3_REL_0006_airbnb002_dim_listings_hosts_listing_id ... OK | 539 + 153 tokens
[07/51] L3_REL_0007_airbnb002_fct_reviews_listing_id ... OK | 539 + 172 tokens
[08/51] L3_REL_0008_airbnb002_fct_reviews_listing_id ... OK | 537 + 208 tokens
[09/51] L3_REL_0009_synthea001_stg_synthea_allergies_patient_id ... OK | 7798 + 380 tokens
[10/51] L3_REL_0010_synthea001_stg_synthea_allergies_encounter_id ... OK | 7802 + 198 tokens
[11/51] L3_REL_0011_synthea001_stg_synthea_careplans_patient_id ... OK | 7798 + 236 tokens
[12/51] L3_REL_0012_synthea001_stg_synthea_careplans_encounter_id ... O

## 7. Haiku adjudication — L3 (Judge V2)


In [15]:
# ============================================================
# RDBScore V2 — L3 Consolidation + Haiku Judge V2
#
# Adjudication semantics:
#
# MATCH:
#   The response identifies the injected source relationship
#   and establishes that its current target is semantically
#   incompatible with the meaning of the source attribute.
#
# PARTIAL_MATCH:
#   The response identifies the injected source relationship
#   as problematic, but does not establish the semantic
#   incompatibility sufficiently.
#
# NO_MATCH:
#   The response does not identify the injected source
#   relationship as problematic.
#
# IMPORTANT:
#   The audited model was NOT shown the original relationship,
#   mutation metadata, or benchmark defect label.
#   Therefore none of these may be required for MATCH.
# ============================================================

import json
import time
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone


JUDGE_MODEL_ID = "anthropic.claude-3-haiku-20240307-v1:0"
JUDGE_PROMPT_VERSION = "RDBSCORE_JUDGE_V2"

L3_RESULTS_DIR = Path(
    "rdbscore_generated_v2/L3/bedrock_results"
)

L3_JUDGE_DIR = Path(
    "rdbscore_generated_v2/L3/judge_haiku_v2"
)

L3_JUDGE_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# JSON recovery — audited model
# ------------------------------------------------------------

def recover_json_l3(text):

    text = (text or "").strip()

    try:
        obj = json.loads(text)

        if (
            isinstance(obj, dict)
            and isinstance(obj.get("findings"), list)
        ):
            return obj, "STRICT_JSON"

    except (ValueError, TypeError):
        pass

    decoder = json.JSONDecoder()

    for i, char in enumerate(text):

        if char != "{":
            continue

        try:
            obj, _ = decoder.raw_decode(text[i:])

            if (
                isinstance(obj, dict)
                and isinstance(obj.get("findings"), list)
            ):
                return obj, "EXTRACTED_JSON"

        except (ValueError, TypeError):
            continue

    return None, "NO_JSON"


# ------------------------------------------------------------
# Consolidate the 51 previously generated Sonnet results
# ------------------------------------------------------------

consolidated_l3 = []
missing_l3 = []

for instance in generation_manifest_l3:

    instance_id = instance["instance_id"]

    result_path = (
        L3_RESULTS_DIR /
        "{}.json".format(instance_id)
    )

    if not result_path.exists():
        missing_l3.append(instance_id)
        continue

    result = json.loads(
        result_path.read_text(encoding="utf-8")
    )

    if result.get("status") != "SUCCESS":
        missing_l3.append(instance_id)
        continue

    gt = json.loads(
        Path(
            instance["ground_truth_path"]
        ).read_text(encoding="utf-8")
    )

    raw_text = (
        result
        .get("llm_result_v2", {})
        .get("text", "")
    )

    parsed, recovery = recover_json_l3(raw_text)

    findings = (
        parsed.get("findings", [])
        if parsed is not None
        else []
    )

    consolidated_l3.append({
        "instance_id": instance_id,
        "level": "L3",
        "operator_id": instance["operator_id"],
        "project": instance["project"],
        "ground_truth": gt,
        "llm_result_v2": result["llm_result_v2"],
        "json_recovery": recovery,
        "parsed_response": parsed,
        "findings": findings,
        "has_findings": len(findings) > 0
    })


assert not missing_l3, (
    "Missing or failed results: {}".format(missing_l3)
)

assert len(consolidated_l3) == 51


print("=" * 80)
print("L3 CONSOLIDATION")
print("=" * 80)

print("Instances:", len(consolidated_l3))

print(
    "JSON recovery:",
    dict(Counter(
        r["json_recovery"]
        for r in consolidated_l3
    ))
)

print(
    "Instances with findings:",
    sum(r["has_findings"] for r in consolidated_l3)
)

print(
    "Total findings:",
    sum(
        len(r["findings"])
        for r in consolidated_l3
    )
)


# ------------------------------------------------------------
# Corrected judge protocol
# ------------------------------------------------------------

JUDGE_SYSTEM_L3_V2 = """
You are an independent evaluator of a database schema auditing
benchmark.

Your task is NOT to audit the schema independently.

You must determine whether the model detected the specific
relationship defect injected into the schema.

The audited model was shown only the mutated schema metadata.
It was NOT shown:
- the original relationship;
- the benchmark mutation;
- the benchmark defect label;
- the ground-truth terminology.

Therefore, NEVER require the model to:
- name the benchmark defect category;
- use the phrase "Semantic Relationship Mismatch";
- mention that a mutation occurred;
- identify the original relationship;
- identify the original target entity or key.

Equivalent natural-language explanations are fully acceptable.

Classify the response using the following criteria.

MATCH:
At least one finding identifies the injected SOURCE relationship
and establishes that its CURRENT target is semantically
incompatible with the meaning or intended role of the source
attribute.

Examples include explanations such as:
- patient_id should not reference a location entity;
- encounter_id should reference an encounter entity, not a person;
- an insurance identifier should reference an insurance/payer
  entity rather than an observation entity.

The response does NOT need to know the original relationship.

PARTIAL_MATCH:
At least one finding identifies the injected SOURCE relationship
as problematic, but does not sufficiently establish semantic
incompatibility.

For example, it may:
- identify the exact source relationship but only say that the
  target table is absent from the supplied metadata;
- identify the exact source relationship as suspicious without
  explaining why the source and target concepts are incompatible.

NO_MATCH:
No finding identifies the injected SOURCE relationship as
problematic.

A finding about another column, relationship, table, datatype,
or semantic issue elsewhere in the schema is NO_MATCH, even if
it is a valid semantic observation.

General statements about "semantic incoherence" are NOT sufficient
unless they concern the injected source relationship.

Additional unrelated findings do not affect classification if one
finding correctly identifies the injected relationship.

Return ONLY valid JSON:

{
  "classification": "MATCH | PARTIAL_MATCH | NO_MATCH",
  "matched_finding_index": null,
  "justification": "Brief explanation"
}
"""


# ------------------------------------------------------------
# Judge JSON recovery
# ------------------------------------------------------------

def recover_judge_json_l3_v2(text):

    text = (text or "").strip()

    try:
        obj = json.loads(text)

        if (
            isinstance(obj, dict)
            and obj.get("classification") in {
                "MATCH",
                "PARTIAL_MATCH",
                "NO_MATCH"
            }
        ):
            return obj, "STRICT_JSON"

    except (ValueError, TypeError):
        pass

    decoder = json.JSONDecoder()

    for i, char in enumerate(text):

        if char != "{":
            continue

        try:
            obj, _ = decoder.raw_decode(text[i:])

            if (
                isinstance(obj, dict)
                and obj.get("classification") in {
                    "MATCH",
                    "PARTIAL_MATCH",
                    "NO_MATCH"
                }
            ):
                return obj, "EXTRACTED_JSON"

        except (ValueError, TypeError):
            continue

    return None, "NO_JSON"


# ------------------------------------------------------------
# Haiku invocation
# ------------------------------------------------------------

def invoke_haiku_judge_l3_v2(record):

    gt = record["ground_truth"]

    # The judge receives GT because its task is evaluation,
    # not independent schema auditing.
    payload = {
        "operator_id": record["operator_id"],
        "level": record["level"],

        "injected_source": {
            "model": gt["source_model"],
            "column": gt["source_column"]
        },

        "current_mutated_relationship": {
            "target_entity":
                gt["mutation"]["mutated_target"],

            "target_key":
                gt["mutation"]["mutated_target_key"]
        },

        "model_findings":
            record["findings"]
    }

    body = {
        "anthropic_version":
            "bedrock-2023-05-31",

        "max_tokens":
            500,

        "temperature":
            0,

        "system":
            JUDGE_SYSTEM_L3_V2,

        "messages": [
            {
                "role": "user",
                "content": json.dumps(
                    payload,
                    ensure_ascii=False
                )
            }
        ]
    }

    t0 = time.time()

    response = bedrock.invoke_model(
        modelId=JUDGE_MODEL_ID,
        body=json.dumps(body),
        contentType="application/json",
        accept="application/json"
    )

    result = json.loads(
        response["body"].read()
    )

    text = "".join(
        block.get("text", "")
        for block in result.get("content", [])
        if block.get("type") == "text"
    )

    parsed, recovery = (
        recover_judge_json_l3_v2(text)
    )

    return {
        "judgment": parsed,
        "raw_response": text,
        "json_recovery": recovery,
        "usage": result.get("usage", {}),
        "latency_seconds":
            time.time() - t0
    }


# ------------------------------------------------------------
# Run corrected adjudication
# ------------------------------------------------------------

judge_errors_l3_v2 = []

for i, record in enumerate(
    consolidated_l3,
    start=1
):

    instance_id = record["instance_id"]

    judge_path = (
        L3_JUDGE_DIR /
        "{}.json".format(instance_id)
    )

    # No findings -> deterministic NO_MATCH.
    if not record["has_findings"]:

        record["judge"] = {
            "classification": "NO_MATCH",
            "matched_finding_index": None,
            "justification":
                "No findings returned.",
            "adjudication":
                "AUTOMATIC_EMPTY"
        }

        continue

    # Resume if V2 judgment already exists.
    if judge_path.exists():

        previous = json.loads(
            judge_path.read_text(
                encoding="utf-8"
            )
        )

        judgment = previous.get(
            "judgment", {}
        )

        if judgment.get(
            "classification"
        ) in {
            "MATCH",
            "PARTIAL_MATCH",
            "NO_MATCH"
        }:

            record["judge"] = judgment
            record["judge"][
                "adjudication"
            ] = "HAIKU_V2"

            continue

    try:

        result = invoke_haiku_judge_l3_v2(
            record
        )

        if result["judgment"] is None:
            raise ValueError(
                "Judge returned no recoverable JSON"
            )

        judge_path.write_text(
            json.dumps(
                result,
                indent=2,
                ensure_ascii=False
            ),
            encoding="utf-8"
        )

        record["judge"] = (
            result["judgment"]
        )

        record["judge"][
            "adjudication"
        ] = "HAIKU_V2"

        print(
            "[{:02d}/51] {}: {}".format(
                i,
                instance_id,
                result["judgment"][
                    "classification"
                ]
            )
        )

    except Exception as e:

        judge_errors_l3_v2.append({
            "instance_id":
                instance_id,
            "error":
                str(e)
        })

        print(
            "[{:02d}/51] {}: ERROR {}".format(
                i,
                instance_id,
                e
            )
        )


# ------------------------------------------------------------
# Final L3 dataset
# ------------------------------------------------------------

scored_l3_v2 = [
    r for r in consolidated_l3
    if "judge" in r
]

counts_l3_v2 = Counter(
    r["judge"]["classification"]
    for r in scored_l3_v2
)

N_L3 = len(consolidated_l3)


summary_l3_v2 = {
    "protocol_version":
        "RDBSCORE_V2",

    "level":
        "L3",

    "judge_model_id":
        JUDGE_MODEL_ID,

    "judge_prompt_version":
        JUDGE_PROMPT_VERSION,

    "instances":
        N_L3,

    "scored":
        len(scored_l3_v2),

    "judge_errors":
        len(judge_errors_l3_v2),

    "classification_counts":
        dict(counts_l3_v2),

    "strict_detection_rate":
        counts_l3_v2["MATCH"] / N_L3,

    "match_or_partial_rate":
        (
            counts_l3_v2["MATCH"]
            + counts_l3_v2["PARTIAL_MATCH"]
        ) / N_L3
}


scored_path = (
    L3_JUDGE_DIR /
    "rdbscore_experiment_v2_l3_scored.json"
)

scored_path.write_text(
    json.dumps(
        scored_l3_v2,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)


summary_path = (
    L3_JUDGE_DIR /
    "judge_batch_summary_l3.json"
)

summary_path.write_text(
    json.dumps(
        summary_l3_v2,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)


print("\n" + "=" * 80)
print("L3 JUDGE V2 COMPLETE")
print("=" * 80)

print(
    json.dumps(
        summary_l3_v2,
        indent=2,
        ensure_ascii=False
    )
)

print("\nScored dataset:", scored_path)
print("Summary:", summary_path)

L3 CONSOLIDATION
Instances: 51
JSON recovery: {'STRICT_JSON': 14, 'EXTRACTED_JSON': 37}
Instances with findings: 51
Total findings: 89
[01/51] L3_REL_0001_airbnb001_dim_listings_hosts_host_id: MATCH
[02/51] L3_REL_0002_airbnb001_dim_listings_hosts_listing_id: MATCH
[03/51] L3_REL_0003_airbnb001_fct_reviews_listing_id: MATCH
[04/51] L3_REL_0004_airbnb001_fct_reviews_listing_id: PARTIAL_MATCH
[05/51] L3_REL_0005_airbnb002_dim_listings_hosts_host_id: MATCH
[06/51] L3_REL_0006_airbnb002_dim_listings_hosts_listing_id: MATCH
[07/51] L3_REL_0007_airbnb002_fct_reviews_listing_id: MATCH
[08/51] L3_REL_0008_airbnb002_fct_reviews_listing_id: PARTIAL_MATCH
[09/51] L3_REL_0009_synthea001_stg_synthea_allergies_patient_id: PARTIAL_MATCH
[10/51] L3_REL_0010_synthea001_stg_synthea_allergies_encounter_id: PARTIAL_MATCH
[11/51] L3_REL_0011_synthea001_stg_synthea_careplans_patient_id: PARTIAL_MATCH
[12/51] L3_REL_0012_synthea001_stg_synthea_careplans_encounter_id: MATCH
[13/51] L3_REL_0013_synthea001_stg_

## 8. Provenance validation and final V2 baseline

This section validates one-to-one correspondence between the freshly generated Sonnet outputs and Haiku judgments, then consolidates the complete 1,561-instance RDBScore V2 LLM-only baseline.


## 9. V2 frozen baseline

The outputs of this notebook constitute the **RDBScore V2 LLM-only baseline** for the clean final run. The benchmark instances are assumed to have been freshly generated by the exploration/construction notebook before this execution.


In [16]:
# ============================================================
# RDBScore V2 — Provenance Validation + Final LLM-Only Baseline
#
# Clean final-run invariants:
#   - 1510 unique L1/L2 Sonnet records
#   - every L1/L2 response with findings has exactly one SUCCESS judge
#   - judge.model_findings exactly match Sonnet findings
#   - 51 scored L3 records
#   - 1561 unique final benchmark records
# ============================================================

import json
from pathlib import Path
from collections import Counter


# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

JUDGE_L12_DIR = Path(
    "rdbscore_generated_v2/judge_haiku_v1"
)

L3_SCORED_PATH = Path(
    "rdbscore_generated_v2/L3/judge_haiku_v2/"
    "rdbscore_experiment_v2_l3_scored.json"
)

FINAL_DIR = Path(
    "rdbscore_generated_v2/final_results"
)

FINAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# Validate L1/L2 Sonnet records
# ------------------------------------------------------------

assert len(records_v2) == 1510, (
    "Expected 1510 L1/L2 Sonnet records, found {}"
    .format(len(records_v2))
)

record_ids = [
    r["mutation_id"]
    for r in records_v2
]

assert len(set(record_ids)) == 1510, (
    "Duplicate mutation_id found in L1/L2 Sonnet records."
)

record_by_id = {
    r["mutation_id"]: r
    for r in records_v2
}


# ------------------------------------------------------------
# Load all successful L1/L2 judge records
# ------------------------------------------------------------

judge_l12 = {}

for path in sorted(JUDGE_L12_DIR.glob("*.judge.json")):

    obj = json.loads(
        path.read_text(encoding="utf-8")
    )

    if obj.get("status") != "SUCCESS":
        continue

    mutation_id = obj.get("mutation_id")

    if not mutation_id:
        raise ValueError(
            "SUCCESS judge file without mutation_id: {}"
            .format(path)
        )

    if mutation_id in judge_l12:
        raise ValueError(
            "Duplicate judge result for {}"
            .format(mutation_id)
        )

    judge_l12[mutation_id] = obj


# ------------------------------------------------------------
# Judge coverage
# ------------------------------------------------------------

cases_with_findings = {
    r["mutation_id"]
    for r in records_v2
    if r.get("findings")
}

judged_ids = set(judge_l12.keys())

missing_ids = sorted(
    cases_with_findings - judged_ids
)

extra_ids = sorted(
    judged_ids - cases_with_findings
)

print("=" * 100)
print("RDBSCORE V2 — L1/L2 PROVENANCE VALIDATION")
print("=" * 100)

print("Sonnet records       :", len(records_v2))
print("Cases with findings  :", len(cases_with_findings))
print("SUCCESS judge records:", len(judged_ids))
print("Missing judgments    :", len(missing_ids))
print("Extra judgments      :", len(extra_ids))

assert not missing_ids, (
    "{} Sonnet cases with findings have no SUCCESS judge."
    .format(len(missing_ids))
)

assert not extra_ids, (
    "{} SUCCESS judges do not correspond to a Sonnet case with findings."
    .format(len(extra_ids))
)


# ------------------------------------------------------------
# Judge / Sonnet finding consistency
# ------------------------------------------------------------

inconsistent = []

for mutation_id in sorted(judged_ids):

    judge_record = judge_l12[mutation_id]
    sonnet_record = record_by_id[mutation_id]

    judge_findings = (
        judge_record.get("model_findings")
        or []
    )

    sonnet_findings = (
        sonnet_record.get("findings")
        or []
    )

    judge_canonical = json.dumps(
        judge_findings,
        sort_keys=True,
        ensure_ascii=False
    )

    sonnet_canonical = json.dumps(
        sonnet_findings,
        sort_keys=True,
        ensure_ascii=False
    )

    if judge_canonical != sonnet_canonical:
        inconsistent.append(mutation_id)


print("Inconsistent pairs   :", len(inconsistent))

assert not inconsistent, (
    "{} judge/Sonnet finding pairs are inconsistent."
    .format(len(inconsistent))
)

print("L1/L2 provenance validation: PASS")


# ------------------------------------------------------------
# Build complete L1/L2 scored dataset
# ------------------------------------------------------------

scored_l12 = []

for record in records_v2:

    mutation_id = record["mutation_id"]
    findings = record.get("findings", [])

    if findings:

        judge_record = judge_l12[mutation_id]

        judgment = (
            judge_record.get("judgment")
            or judge_record.get("parsed")
            or judge_record.get("judge", {}).get("parsed")
        )

        if judgment is None and judge_record.get(
            "classification"
        ) in {
            "MATCH",
            "PARTIAL_MATCH",
            "NO_MATCH"
        }:
            judgment = judge_record

        if not judgment:
            raise ValueError(
                "No judgment payload found in {}"
                .format(mutation_id)
            )

        classification = judgment["classification"]

        assert classification in {
            "MATCH",
            "PARTIAL_MATCH",
            "NO_MATCH"
        }, (
            "Invalid classification for {}: {}"
            .format(mutation_id, classification)
        )

        adjudication = "HAIKU_V1"
        justification = judgment.get("justification")
        matched_finding_index = judgment.get(
            "matched_finding_index"
        )

    else:

        classification = "NO_MATCH"
        adjudication = "AUTOMATIC_EMPTY"
        justification = (
            "The audited model returned no findings."
        )
        matched_finding_index = None

    scored = dict(record)

    scored["judge"] = {
        "classification": classification,
        "matched_finding_index": matched_finding_index,
        "justification": justification,
        "adjudication": adjudication
    }

    scored_l12.append(scored)


assert len(scored_l12) == 1510


# ------------------------------------------------------------
# Load and validate final L3 dataset
# ------------------------------------------------------------

assert L3_SCORED_PATH.exists(), (
    "L3 scored dataset not found: {}"
    .format(L3_SCORED_PATH)
)

scored_l3 = json.loads(
    L3_SCORED_PATH.read_text(
        encoding="utf-8"
    )
)

assert len(scored_l3) == 51, (
    "Expected 51 L3 cases, found {}"
    .format(len(scored_l3))
)

l3_ids = [
    r["instance_id"]
    for r in scored_l3
]

assert len(set(l3_ids)) == 51, (
    "Duplicate instance_id found in L3 scored records."
)

for r in scored_l3:
    classification = r["judge"]["classification"]

    assert classification in {
        "MATCH",
        "PARTIAL_MATCH",
        "NO_MATCH"
    }, (
        "Invalid L3 classification for {}: {}"
        .format(r["instance_id"], classification)
    )


# ------------------------------------------------------------
# Normalize records for final analysis
# ------------------------------------------------------------

final_records = []

for r in scored_l12:

    final_records.append({
        "mutation_id": r["mutation_id"],
        "level": r["level"],
        "operator_id": r["operator_id"],
        "project": r["project"],
        "classification": r["judge"]["classification"],
        "adjudication": r["judge"]["adjudication"]
    })


for r in scored_l3:

    final_records.append({
        "mutation_id": r["instance_id"],
        "level": "L3",
        "operator_id": r["operator_id"],
        "project": r["project"],
        "classification": r["judge"]["classification"],
        "adjudication": r["judge"].get(
            "adjudication",
            "HAIKU_V2"
        )
    })


assert len(final_records) == 1561, (
    "Expected 1561 final records, found {}"
    .format(len(final_records))
)

final_ids = [
    r["mutation_id"]
    for r in final_records
]

assert len(set(final_ids)) == 1561, (
    "Duplicate mutation_id found in final dataset."
)

print("L3 scored records    :", len(scored_l3))
print("Final records        :", len(final_records))
print("Final uniqueness     : PASS")


# ------------------------------------------------------------
# Aggregate
# ------------------------------------------------------------

def summarize(records):

    counts = Counter(
        r["classification"]
        for r in records
    )

    n = len(records)

    match = counts["MATCH"]
    partial = counts["PARTIAL_MATCH"]
    no_match = counts["NO_MATCH"]

    assert match + partial + no_match == n

    return {
        "N": n,
        "MATCH": match,
        "PARTIAL_MATCH": partial,
        "NO_MATCH": no_match,
        "strict_detection_rate":
            match / n if n else 0,
        "match_or_partial_rate":
            (match + partial) / n if n else 0
    }


by_level = {}

for level in ["L1", "L2", "L3"]:

    subset = [
        r for r in final_records
        if r["level"] == level
    ]

    by_level[level] = summarize(subset)


by_operator = {}

operators = sorted({
    r["operator_id"]
    for r in final_records
})

for operator in operators:

    subset = [
        r for r in final_records
        if r["operator_id"] == operator
    ]

    by_operator[operator] = summarize(subset)


overall = summarize(final_records)


# ------------------------------------------------------------
# Final report
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("RDBSCORE V2 — FINAL LLM-ONLY BASELINE")
print("=" * 100)

print("\nBY LEVEL")

for level in ["L1", "L2", "L3"]:

    s = by_level[level]

    print(
        "{} | N={:4d} | "
        "MATCH={:4d} | "
        "PARTIAL={:4d} | "
        "NO_MATCH={:4d} | "
        "Strict={:6.2%} | "
        "M+P={:6.2%}"
        .format(
            level,
            s["N"],
            s["MATCH"],
            s["PARTIAL_MATCH"],
            s["NO_MATCH"],
            s["strict_detection_rate"],
            s["match_or_partial_rate"]
        )
    )


print("\nBY OPERATOR")

for operator, s in by_operator.items():

    print(
        "{} | N={:4d} | "
        "MATCH={:4d} | "
        "PARTIAL={:4d} | "
        "NO_MATCH={:4d} | "
        "Strict={:6.2%} | "
        "M+P={:6.2%}"
        .format(
            operator,
            s["N"],
            s["MATCH"],
            s["PARTIAL_MATCH"],
            s["NO_MATCH"],
            s["strict_detection_rate"],
            s["match_or_partial_rate"]
        )
    )


print("\nOVERALL")

print(
    "TOTAL | N={:4d} | "
    "MATCH={:4d} | "
    "PARTIAL={:4d} | "
    "NO_MATCH={:4d} | "
    "Strict={:6.2%} | "
    "M+P={:6.2%}"
    .format(
        overall["N"],
        overall["MATCH"],
        overall["PARTIAL_MATCH"],
        overall["NO_MATCH"],
        overall["strict_detection_rate"],
        overall["match_or_partial_rate"]
    )
)


# ------------------------------------------------------------
# Save final reproducible artifacts
# ------------------------------------------------------------

final_summary = {
    "protocol_version": "RDBSCORE_V2",
    "experiment": "LLM_ONLY_BASELINE",
    "instances": len(final_records),
    "provenance_validation": {
        "l12_sonnet_records": len(records_v2),
        "l12_cases_with_findings": len(cases_with_findings),
        "l12_success_judges": len(judged_ids),
        "l12_missing_judges": len(missing_ids),
        "l12_extra_judges": len(extra_ids),
        "l12_inconsistent_pairs": len(inconsistent),
        "l3_scored_records": len(scored_l3),
        "final_unique_records": len(set(final_ids))
    },
    "by_level": by_level,
    "by_operator": by_operator,
    "overall": overall
}


(FINAL_DIR / "rdbscore_v2_final_records.json").write_text(
    json.dumps(
        final_records,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)


(FINAL_DIR / "rdbscore_v2_final_summary.json").write_text(
    json.dumps(
        final_summary,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)


print(
    "\nSaved:",
    FINAL_DIR / "rdbscore_v2_final_records.json"
)

print(
    "Saved:",
    FINAL_DIR / "rdbscore_v2_final_summary.json"
)

print("\nRDBScore V2 final-run validation: PASS")

RDBSCORE V2 — L1/L2 PROVENANCE VALIDATION
Sonnet records       : 1510
Cases with findings  : 541
SUCCESS judge records: 541
Missing judgments    : 0
Extra judgments      : 0
Inconsistent pairs   : 0
L1/L2 provenance validation: PASS
L3 scored records    : 51
Final records        : 1561
Final uniqueness     : PASS

RDBSCORE V2 — FINAL LLM-ONLY BASELINE

BY LEVEL
L1 | N=1161 | MATCH=  14 | PARTIAL= 106 | NO_MATCH=1041 | Strict= 1.21% | M+P=10.34%
L2 | N= 349 | MATCH=  18 | PARTIAL= 294 | NO_MATCH=  37 | Strict= 5.16% | M+P=89.40%
L3 | N=  51 | MATCH=  30 | PARTIAL=  21 | NO_MATCH=   0 | Strict=58.82% | M+P=100.00%

BY OPERATOR
L1_REMOVE_NOT_NULL | N= 861 | MATCH=  13 | PARTIAL=  86 | NO_MATCH= 762 | Strict= 1.51% | M+P=11.50%
L1_REMOVE_UNIQUE | N= 300 | MATCH=   1 | PARTIAL=  20 | NO_MATCH= 279 | Strict= 0.33% | M+P= 7.00%
L2_INTRODUCE_TRANSITIVE_DEPENDENCY | N= 349 | MATCH=  18 | PARTIAL= 294 | NO_MATCH=  37 | Strict= 5.16% | M+P=89.40%
L3_INTRODUCE_SEMANTIC_RELATIONSHIP_MISMATCH | N=  